# RAG 评估速查 Notebook

> **配套文档**: `RAG_Evaluation_Deep_Dive.md`  
> **定位**: 精选核心代码，支持 **无 API Key Run All**（Mock 模式）

## 内容
1. 手动实现四大指标（Binary + Weighted 对照）
2. Ranking Metrics：HitRate / MRR / NDCG 完整计算
3. RAGAS 评估 Demo（有 Key 时跑 Live，无 Key 时 Mock）
4. 四象限诊断 + 优化建议生成
5. 速查表（阈值 / 公式 / 优化策略）

---
# Part 1 · 手动实现四大指标（Binary + Weighted 对照）

In [ ]:
import re, json
from math import log2
from typing import Dict, List, Set, Tuple

# ========== 模拟知识库（8 篇文档）==========
KNOWLEDGE_BASE = {
    "doc1": "RAG（检索增强生成）是一种将检索系统与大语言模型结合的技术，用于提升生成质量。",
    "doc2": "向量数据库（如 Chroma、Pinecone）用于存储文档嵌入，支持高效的相似度检索。",
    "doc3": "Chunk Size 是 RAG 中重要的超参数，过大导致噪声增多，过小导致语义不完整。",
    "doc4": "Reranker 模型可以对初步检索结果重新排序，显著提升 Context Precision。",
    "doc5": "LLM 幻觉（Hallucination）是指模型生成了与上下文不一致或凭空捏造的内容。",
    "doc6": "HyDE（假设文档嵌入）是一种 Query 增强技术，先让 LLM 生成假设答案再检索。",
    "doc7": "BM25 是经典的关键词检索算法，与向量检索结合可实现混合检索（Hybrid Search）。",
    "doc8": "RAGAS 是一个开源 RAG 评估框架，提供 Faithfulness、Answer Relevancy 等多个指标。",
}

# ========== 模拟检索器（关键词重叠）==========
def retrieve_by_keyword(query: str, top_k: int = 3) -> list:
    """基于关键词重叠的简易检索器"""
    scores = {}
    for doc_id, content in KNOWLEDGE_BASE.items():
        overlap = sum(1 for c in query if c in content)
        scores[doc_id] = overlap
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [doc_id for doc_id, _ in ranked[:top_k]]

# ========== 评估查询（含手工标注的 Ground Truth）==========
TEST_QUERIES = [
    {"query": "RAG 系统如何通过检索提升生成质量？", "relevant_ids": {"doc1", "doc2", "doc7"}},
    {"query": "如何减少 LLM 幻觉问题？",             "relevant_ids": {"doc5", "doc1"}},
    {"query": "Reranker 和 Chunk Size 对检索的影响",  "relevant_ids": {"doc3", "doc4"}},
]

print(f"知识库: {len(KNOWLEDGE_BASE)} 篇 | 评估查询: {len(TEST_QUERIES)} 条")
print("✅ 环境就绪 (Mock 模式 — 无需 API Key)")

## 1.1 Context Recall & Precision — 同一事例并排对比

**核心认知**：Recall 和 Precision 共享同一个分子（TP = |GT ∩ Retrieved|），但分母不同。

$$\text{Recall} = \frac{TP}{|GT|} \qquad \text{Precision} = \frac{TP}{|Retrieved|}$$

下面用同一组数据同时计算两个指标。

### 示例数据
```
问题："RAG 系统如何通过检索提升生成质量？"
GT 文档 ID：{doc1, doc2, doc7}    ← 该找 3 篇
检索结果：  [doc1, doc4, doc8]    ← 实际找 3 篇
命中：      {doc1}                ← 只有 1 篇同时出现在两边
```

| 指标 | 计算 | 分数 | 解读 |
|------|------|:--:|------|
| **Recall** | 1/3 | **0.33** | doc2、doc7 漏了——该找的没找全 |
| **Precision** | 1/3 | **0.33** | doc4、doc8 是噪声——找的不干净 |

> 本例中 Recall = Precision = 0.33 是巧合（因为 |GT| = |Retrieved| = 3）。实际中两者通常不同——检索 10 篇命中 3 篇，而 GT 只有 4 篇时，Recall = 3/4 = 0.75，Precision = 3/10 = 0.30。

In [ ]:
def compute_recall_binary(retrieved_ids: List[str], relevant_ids: Set[str]) -> float:
    """Binary Context Recall = TP / |GT|"""
    if not relevant_ids:
        return 0.0
    hits = [d for d in retrieved_ids if d in relevant_ids]
    return len(hits) / len(relevant_ids)


def compute_precision_binary(retrieved_ids: List[str], relevant_ids: Set[str]) -> float:
    """Binary Context Precision = TP / |Retrieved|"""
    if not retrieved_ids:
        return 0.0
    hits = [d for d in retrieved_ids if d in relevant_ids]
    return len(hits) / len(retrieved_ids)


# —— 同一数据，并排计算 ——
print("=" * 70)
print("Recall vs Precision — 同一事例并排对比")
print("=" * 70)
print(f"{'查询':<34} {'Recall':>8} {'Precision':>9} {'TP':>4} {'|GT|':>4} {'|Ret|':>4}")
print("-" * 70)

for item in TEST_QUERIES:
    q = item["query"]
    rel = item["relevant_ids"]
    ret = retrieve_by_keyword(q, top_k=3)

    r = compute_recall_binary(ret, rel)
    p = compute_precision_binary(ret, rel)
    tp = len(set(ret) & rel)

    print(f"{q[:32]:<34} {r:>8.3f} {p:>9.3f} {tp:>4} {len(rel):>4} {len(ret):>4}")
    print(f"  GT={rel}  Retrieved={ret}  Hit={set(ret) & rel}")
    if r != p:
        diff = abs(r - p)
        direction = "Recall > Precision → 检索太多但漏了关键文档" if r > p else "Precision > Recall → 检索干净但覆盖面不足"
        print(f"  ⚖️ Recall≠Precision (差{diff:.2f}) → {direction}")
    print()

print("=" * 70)
print("💡 Recall 和 Precision 共享分子 TP，分母不同 → 理解这一点就理解了检索评估的核心")

## 1.2 Context Precision — Weighted 版（RAGAS 默认）

上节的 Binary Precision 有个盲区：**排名位置不影响分数**。如果 [✓, ✗, ✗] 和 [✗, ✗, ✓]，Binary 版都是 1/3 = 0.33——但显然第一种排序更好（用户先看到正确结果）。

Weighted Precision 惩罚「相关文档排在后面」的情况：

$$P@K_{weighted} = \frac{\sum_{i=1}^{K} P@i \cdot rel(C_i)}{\sum_{i=1}^{K} rel(C_i)}$$

In [ ]:
def compute_precision_weighted(retrieved_ids: List[str], relevant_ids: Set[str]) -> float:
    """Weighted Context Precision（RAGAS 默认）—— 按排名位置加权"""
    total_weighted = 0.0
    total_rel = 0
    hits_so_far = 0

    for i, doc_id in enumerate(retrieved_ids, start=1):
        rel = 1 if doc_id in relevant_ids else 0
        if rel:
            hits_so_far += 1
            p_at_i = hits_so_far / i
            total_weighted += p_at_i * rel
            total_rel += 1

    return total_weighted / total_rel if total_rel > 0 else 0.0


# —— 对照实验：Binary vs Weighted ——
print("=" * 75)
print(f"{'查询':<30} {'Binary':>8} {'Weighted':>9} {'Delta':>8}")
print("-" * 75)

for item in TEST_QUERIES:
    q = item["query"]
    rel = item["relevant_ids"]
    ret1 = retrieve_by_keyword(q, top_k=3)   # 标准检索
    ret2 = list(reversed(ret1))               # 故意反转顺序（模拟差排序）

    pb1 = compute_precision_binary(ret1, rel)
    pw1 = compute_precision_weighted(ret1, rel)
    pb2 = compute_precision_binary(ret2, rel)
    pw2 = compute_precision_weighted(ret2, rel)

    print(f"{q[:28]:<30} {pb1:>8.3f} {pw1:>9.3f} {'--':>8}")
    print(f"  {'(故意逆序)':>28} {pb2:>8.3f} {pw2:>9.3f} {pw2-pw1:>+8.3f}")
    print()

print("=" * 75)
print("注意：逆序后 Binary 不变，但 Weighted 大幅下降 → 体现了排序惩罚")
print("RAG 中 LLM 更关注靠前的上下文 → Weighted 版本更符合实际")

## 1.3 Answer Relevancy — Jaccard vs 语义版

**Jaccard 版是演示用的，生产环境不可用。** 真实的 Answer Relevancy 需要 LLM 反向生成问题 + Embedding 计算语义相似度（见 Part 3 RAGAS Demo）。

In [ ]:
def compute_relevancy_jaccard(query: str, answer: str) -> float:
    """
    ⚠️  Jaccard 关键词代理 — 仅演示概念！
    真实评估需要 LLM 反向生成问题 + Embedding 语义相似度
    """
    def tokenize(text: str) -> set:
        return set(re.findall(r"[一-鿿]+|[a-zA-Z0-9]+", text.lower()))
    qt = tokenize(query)
    at = tokenize(answer)
    return len(qt & at) / len(qt | at) if (qt | at) else 0.0


# —— 演示 Jaccard 的局限性 ——
test_pairs = [
    ("什么是 RAG？", "RAG 是检索增强生成技术"),
    ("什么是 RAG？它如何减少幻觉？", "RAG 技术通过检索真实文档有效缓解了 LLM 的幻觉问题"),
    ("如何减少 LLM 幻觉问题？", "RAG 系统从知识库检索相关文档注入上下文"),
]

print("Jaccard Relevancy 演示：")
print("-" * 70)
for q, a in test_pairs:
    score = compute_relevancy_jaccard(q, a)
    print(f"Q: {q[:40]}")
    print(f"A: {a[:40]}")
    print(f"Jaccard: {score:.3f}  {'⚠️ 换种说法就低得离谱！' if score < 0.2 else ''}")
    print()

print("结论：Jaccard 代理几乎不可用 → 生产环境必须用语义相似度（RAGAS AnswerRelevancy）")

## 1.4 Faithfulness — LLM-as-Judge 流程示意

Mock 模式演示 Claim 提取 + 逐条验证的**计算逻辑**（不调用真实 LLM）。

In [ ]:
# ========== MOCK Faithfulness 演示 ==========

MOCK_ANSWER = (
    "RAG 通过检索相关文档并注入上下文，引导模型基于事实回答，"
    "从而有效减少幻觉现象。此外，RAG 还能提升回答的时效性。"
)

MOCK_CONTEXT = [
    "RAG 技术通过在生成前检索相关文档，将外部知识注入模型上下文，减少幻觉。",
    "LLM 幻觉是指模型生成与上下文不一致或凭空捏造的内容。",
]

# 模拟 LLM 拆解的 claims（实际使用中由 LLM 完成）
MOCK_CLAIMS = [
    {"claim": "RAG 通过检索相关文档并注入上下文", "supported": True},
    {"claim": "RAG 引导模型基于事实回答", "supported": True},
    {"claim": "RAG 能有效减少幻觉现象", "supported": True},
    {"claim": "RAG 能提升回答的时效性", "supported": False},  # 上下文没提！
]

faithfulness = sum(1 for c in MOCK_CLAIMS if c["supported"]) / len(MOCK_CLAIMS)

print("=" * 60)
print("Faithfulness Mock 演示 (Claims → NLI → Score)")
print("=" * 60)
print(f"\n📝 答案:\n   {MOCK_ANSWER}\n")
print(f"📚 上下文:\n   {MOCK_CONTEXT[0]}\n   {MOCK_CONTEXT[1]}\n")
print("🔍 Claim 逐条验证:")
for i, c in enumerate(MOCK_CLAIMS, 1):
    status = "✅ 支撑" if c["supported"] else "❌ 无支撑"
    print(f"   Claim {i}: {status} — {c['claim']}")

print(f"\n📊 Faithfulness = {faithfulness:.2f} = 3/4（3 条被上下文支撑，1 条无依据）")
print(f"\n⚠️  Claim 4（时效性）是 LLM 自己编造的——这就是 Faithfulness 要抓的问题")
print("=" * 60)

In [ ]:
# ========== NLI 三分类 + PARTIALLY_SUPPORTED Demo ==========
# Mock: 演示三分法判断逻辑

MOCK_NLI_RESULTS = [
    {"claim": "RAG 技术通过检索真实文档减少幻觉", "verdict": "SUPPORTED",
     "evidence": "上下文：'RAG技术...减少幻觉'", "reason": "上下文有明确等价表述"},
    {"claim": "RAG 能提升回答的时效性", "verdict": "PARTIALLY_SUPPORTED",
     "evidence": "上下文没有提到'时效性'", "reason": "前半句有支撑，时效性部分无依据"},
    {"claim": "RAG 于 2015 年由 OpenAI 发明", "verdict": "NOT_SUPPORTED",
     "evidence": "上下文未提及任何发明时间和机构", "reason": "完全无依据，典型幻觉"},
]

print("=" * 65)
print("Faithfulness 三分法验证演示 (NLI-style)")
print("=" * 65)
print(f"\n{'Claim':<42} {'判定':>22}")
print("-" * 65)
for r in MOCK_NLI_RESULTS:
    verdict_icon = {"SUPPORTED": "✅", "PARTIALLY_SUPPORTED": "⚠️", "NOT_SUPPORTED": "❌"}
    print(f"  {r['claim']:<40} {verdict_icon[r['verdict']]} {r['verdict']:<18}")
    print(f"    依据: {r['evidence'][:60]}")

supported_count = sum(1 for r in MOCK_NLI_RESULTS if r['verdict'] == 'SUPPORTED')
# PARTIALLY_SUPPORTED 在严格模式下算 0，宽松模式下算 0.5
faithfulness_strict = supported_count / len(MOCK_NLI_RESULTS)
faithfulness_lenient = (supported_count + 0.5) / len(MOCK_NLI_RESULTS)

print(f"\n📊 Faithfulness (严格模式) = {faithfulness_strict:.2f}")
print(f"📊 Faithfulness (宽松模式, PARTIALLY=0.5) = {faithfulness_lenient:.2f}")
print("=" * 65)

## 1.5 Faithfulness 进阶：NLI三分类 + PARTIALLY_SUPPORTED三分法

### NLI 三分类模型

Faithfulness 验证底层依赖**自然语言推理（NLI）**，将 Claim 与 Context 的关系分为三类：

| 关系 | 含义 | Faithfulness 中 |
|------|------|:--:|
| **Entailment（蕴含）** | Context 逻辑上支持 Claim | ✅ supported |
| **Contradiction（矛盾）** | Context 反驳 Claim | ❌ 幻觉 |
| **Neutral（中立）** | Context 不支持也不反驳 | ❌ 无法验证 |

### PARTIALLY_SUPPORTED 三分法

实际评估中，二分类（支持/不支持）不够精细。lesson2 引入了三分法：

| 判断 | 标准 | 示例 |
|------|------|------|
| **SUPPORTED** | 上下文中有明确等价表述 | "RAG减少幻觉" → ✅ 上下文第2段写了 |
| **NOT_SUPPORTED** | 上下文完全找不到证据 | "支持部分退款" → ❌ 上下文从未提及 |
| **PARTIALLY_SUPPORTED** | 部分有依据但超出范围 | "RAG减少幻觉且提升时效性" → ⚠️ 时效性部分无依据 |

> 面试要点：NLI vs LLM-as-Judge 的取舍——英文用 NLI 初筛 + LLM 复审，中文以 LLM-Judge 为主。

---
# Part 2 · Ranking Metrics：HitRate / P@k / R@k / MRR / NDCG

In [ ]:
# ========== 模拟检索排序数据 ==========
EVAL_CASES = [
    {
        "query": "什么是 RAG 技术？",
        "retrieved": ["doc1", "doc8", "doc4", "doc2", "doc6"],
        "relevance": {"doc1": 3, "doc2": 2, "doc8": 1},   # 3=强相关 2=中 1=弱
    },
    {
        "query": "如何减少 LLM 幻觉？",
        "retrieved": ["doc3", "doc5", "doc1", "doc7", "doc8"],
        "relevance": {"doc1": 3, "doc5": 2},
    },
    {
        "query": "Reranker 有什么作用？",
        "retrieved": ["doc7", "doc2", "doc4", "doc3", "doc8"],
        "relevance": {"doc4": 3, "doc3": 1},
    },
    {
        "query": "什么是 Hybrid Search？",
        "retrieved": ["doc5", "doc8", "doc1", "doc2", "doc7"],
        "relevance": {"doc7": 3, "doc2": 1},
    },
]


def relevant_ids(case: dict) -> set:
    return {d for d, grade in case["relevance"].items() if grade > 0}


# ========== 五大指标函数 ==========

def hit_rate_at_k(case: dict, k: int) -> float:
    """top-k 中是否至少命中 1 个相关文档"""
    top_k = case["retrieved"][:k]
    return 1.0 if set(top_k) & relevant_ids(case) else 0.0


def precision_at_k(case: dict, k: int) -> float:
    """top-k 中相关文档占比"""
    top_k = case["retrieved"][:k]
    return sum(1 for d in top_k if d in relevant_ids(case)) / k if k else 0.0


def recall_at_k(case: dict, k: int) -> float:
    """相关文档中找回了多少"""
    rel = relevant_ids(case)
    if not rel:
        return 0.0
    top_k = case["retrieved"][:k]
    return sum(1 for d in top_k if d in rel) / len(rel)


def reciprocal_rank_at_k(case: dict, k: int) -> float:
    """第一个相关文档排名的倒数"""
    rel = relevant_ids(case)
    for rank, doc_id in enumerate(case["retrieved"][:k], start=1):
        if doc_id in rel:
            return 1.0 / rank
    return 0.0


def ndcg_at_k(case: dict, k: int) -> float:
    """NDCG@k = DCG / IDCG"""
    dcg = 0.0
    for rank, doc_id in enumerate(case["retrieved"][:k], start=1):
        gain = case["relevance"].get(doc_id, 0)
        dcg += gain / log2(rank + 1)

    # IDCG = 理想排序下的 DCG（所有高相关排最前面）
    ideal_gains = sorted(case["relevance"].values(), reverse=True)[:k]
    idcg = sum(g / log2(i + 2) for i, g in enumerate(ideal_gains))

    return dcg / idcg if idcg > 0 else 0.0


def summarize_at_k(cases: list, k: int) -> dict:
    """总结多条 case 的指标均值"""
    n = len(cases)
    return {
        "hit_rate": sum(hit_rate_at_k(c, k) for c in cases) / n,
        "precision": sum(precision_at_k(c, k) for c in cases) / n,
        "recall": sum(recall_at_k(c, k) for c in cases) / n,
        "mrr": sum(reciprocal_rank_at_k(c, k) for c in cases) / n,
        "ndcg": sum(ndcg_at_k(c, k) for c in cases) / n,
    }


# ========== 运行评估 — 逐条 + k 对比 ==========

print("=" * 80)
print("逐条查询明细 (@k=3)")
print("=" * 80)
print(f"{'查询':<22} {'Top-3':<32} {'Hit':>4} {'P@k':>6} {'R@k':>6} {'RR':>6} {'NDCG':>6}")
print("-" * 80)

for case in EVAL_CASES:
    top3 = ",".join(case["retrieved"][:3])
    print(
        f"{case['query'][:20]:<22} {top3:<32} "
        f"{hit_rate_at_k(case, 3):>4.0f} "
        f"{precision_at_k(case, 3):>6.2f} "
        f"{recall_at_k(case, 3):>6.2f} "
        f"{reciprocal_rank_at_k(case, 3):>6.2f} "
        f"{ndcg_at_k(case, 3):>6.2f}"
    )

# k=1/3/5 对比
print("\n" + "=" * 65)
print("Top-k 配置对比（均值）")
print("=" * 65)
print(f"{'k':>3} {'HitRate':>8} {'Precision':>9} {'Recall':>7} {'MRR':>7} {'NDCG':>7}")
print("-" * 50)

for k in [1, 3, 5]:
    s = summarize_at_k(EVAL_CASES, k)
    print(
        f"{k:>3} {s['hit_rate']:>8.2f} {s['precision']:>9.2f} "
        f"{s['recall']:>7.2f} {s['mrr']:>7.2f} {s['ndcg']:>7.2f}"
    )

print("\n解读：")
print("  k 从 1→5：Recall 明显上升 → 说明 top_k 太小会漏召回")
print("  Precision 不一定上升 → 扩大 k 可能引入更多噪声")
print("  NDCG 缓慢上升 → 相关文档虽然找到了，但排序还可以优化")
print("=" * 65)

---
# Part 3 · RAGAS 评估 Demo

需要 `pip install ragas langchain-openai langchain-community`。  
**无 API Key → 使用 Mock 分数自动跳过 LLM 调用。**

In [ ]:
# ========== 配置：修改这里来切换 Mock / Live ==========
USE_MOCK = True                           # 无 Key 时设为 True
DEEPSEEK_API_KEY = "sk-your-api-key-here"  # 有 Key 时替换
DEEPSEEK_BASE_URL = "https://api.deepseek.com"
# ========================================================

RAGAS_AVAILABLE = False
try:
    from ragas import evaluate, EvaluationDataset
    from ragas.metrics import (
        LLMContextRecall,
        LLMContextPrecisionWithoutReference,
        AnswerRelevancy,
        Faithfulness,
    )
    from ragas.llms import LangchainLLMWrapper
    from ragas.embeddings import LangchainEmbeddingsWrapper
    from langchain_openai import ChatOpenAI
    from langchain_community.embeddings import HuggingFaceEmbeddings
    RAGAS_AVAILABLE = True
except ImportError:
    print("⚠️  RAGAS 未安装。运行: pip install ragas langchain-openai langchain-community")
    print("   将使用 Mock 模式……")

print(f"RAGAS 可用: {RAGAS_AVAILABLE} | Mock 模式: {USE_MOCK}")

In [ ]:
# ========== Langfuse 集成示例（需要 pip install langfuse） ==========

LANGFUSE_AVAILABLE = False
try:
    from langfuse import get_client, propagate_attributes, observe
    LANGFUSE_AVAILABLE = True
except ImportError:
    pass

print("=" * 65)
print("Langfuse 可观测性 (Mock 演示)")
print("=" * 65)

if LANGFUSE_AVAILABLE:
    print("✅ Langfuse 可用")
    print("""
# 接入方式 1: LangChain CallbackHandler（一行接入）
from langfuse.langchain import CallbackHandler
handler = CallbackHandler()
llm = ChatOpenAI(model="gpt-4o", callbacks=[handler])
chain.invoke(input, config={"callbacks": [handler]})

# 接入方式 2: 业务 ID 关联（可复现 trace_id）
trace_id = Langfuse.create_trace_id(seed=order_id)

# 接入方式 3: @observe 装饰器（包装非 LangChain 逻辑）
@observe()
def my_retriever(query):
    return search(query)
""")
else:
    print("⚠️  Langfuse 未安装。运行: pip install langfuse")
    print()
    print("Mock 演示 — Langfuse 三层架构:")
    print("  Layer 1 (实时监控) → Langfuse Dashboard 实时看板")
    print("  Layer 2 (定期评估) → 每周 RAGAS 结果写入 Langfuse Metrics")
    print("  Layer 3 (发布门禁) → CI 中 RAGAS evaluate() < 阈值 → 阻止发布")
    print()
    print("📊 Langfuse Metrics 查询 (GraphQL):")
    print('  metrics=[{"measure":"count","aggregation":"count"}]')
    print('  dimensions=[{"field":"name"}]')
    print("  → 按 Trace name 聚合统计调用次数")

print()
print("RAGAS + Langfuse 互补关系:")
print("  RAGAS  → 离线算分（离线）")
print("  Langfuse → 全链路可观测（在线）")
print("  两者结合 = 评估 + 可观测性全覆盖")
print("=" * 65)

---
# Part 3C · Langfuse 可观测性集成

Langfuse 是开源 LLM 工程平台，提供 Trace、Prompt 管理、Metrics。与 RAGAS 互补——RAGAS 离线算分，Langfuse 在线看全链路。

**核心概念**：

| 概念 | 作用 |
|------|------|
| **Trace** | 一次完整请求链路（提问→检索→生成） |
| **Span** | Trace 中的子步骤（单次检索/LLM调用） |
| **Session** | 多轮对话或同一用户会话 |
| **Prompt 管理** | 版本化存储拉取 Prompt，支持A/B对比 |

In [ ]:
# ========== TestsetGenerator Mock 演示 ==========
# 展示自动生成的测试集数据格式（无需 API Key）

MOCK_GENERATED_TESTSET = [
    {
        "user_input": "Transformer 架构的核心创新是什么？",
        "reference": "Transformer 的核心创新是完全基于注意力机制，摒弃了循环和卷积结构。",
        "reference_contexts": ["Transformer 架构由 Vaswani 等人在 2017 年提出，完全基于注意力机制..."],
        "synthesizer": "SingleHopSpecific",
    },
    {
        "user_input": "RAG 技术如何同时解决检索精度和生成幻觉两个问题？",
        "reference": "RAG 通过检索相关文档注入上下文来减少幻觉，同时通过 Reranker 提升检索精度。",
        "reference_contexts": ["RAG 技术...减少幻觉", "Reranker 模型可以...提升 Context Precision"],
        "synthesizer": "MultiHopSpecific",
    },
]

print("=" * 70)
print("RAGAS TestsetGenerator 自动生成测试集 (Mock 演示)")
print("=" * 70)
for i, sample in enumerate(MOCK_GENERATED_TESTSET, 1):
    print(f"\n样本 {i} [{sample['synthesizer']}]")
    print(f"  Q: {sample['user_input']}")
    print(f"  A: {sample['reference'][:60]}...")
    print(f"  参考上下文数: {len(sample['reference_contexts'])}")
    print(f"  → {sample['reference_contexts'][0][:50]}...")

print("\n" + "=" * 70)
print("自定义生成 vs RAGAS 官方生成:")
print("  官方: 全自动 + Persona + Graph → 多样性高，逻辑不透明")
print("  自定义: LLM逐Chunk生成 → 精确控制 reference_contexts，逻辑透明")
print("  推荐: 官方跑基线 + 自定义补特定领域QA")
print("=" * 70)

---
# Part 3B · RAGAS TestsetGenerator 自动测试集生成

从原始文档自动生成评估用的 QA 对。**Mock 模式：演示数据格式和流程。**

```python
# 完整流程（需要 API Key）
from ragas.testset import TestsetGenerator
from ragas.testset.synthesizers import (
    SingleHopSpecificQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer,
    MultiHopAbstractQuerySynthesizer,
)

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
testset = generator.generate_with_langchain_docs(
    chunks,
    testset_size=50,
    query_distribution=[
        (SingleHopSpecificQuerySynthesizer(), 0.5),   # 50% 简单单跳
        (MultiHopSpecificQuerySynthesizer(), 0.25),    # 25% 多跳推理
        (MultiHopAbstractQuerySynthesizer(), 0.25),    # 25% 多跳抽象
    ]
)
# 每条样本自动包含: user_input, reference, reference_contexts
```

In [ ]:
# ========== RAGAS 样本数据 ==========
EVAL_SAMPLES = [
    {
        "user_input": "什么是 RAG，它的主要作用是什么？",
        "retrieved_contexts": [
            "检索增强生成（RAG）是将信息检索与大语言模型结合的技术。"
            "RAG 先从知识库检索相关文档，再将其作为上下文提供给 LLM 生成答案，从而减少幻觉并提升答案准确性。",
            "大语言模型的幻觉问题是指模型生成听起来合理但实际不准确的内容。"
            "RAG 技术通过提供真实文档作为依据，有效缓解了幻觉问题。",
        ],
        "response": "RAG（检索增强生成）是一种结合信息检索与大语言模型的技术。它的主要作用是通过检索真实文档来减少 LLM 生成虚假信息（幻觉），提升答案的准确性。",
        "reference": "RAG 是检索增强生成技术，将检索与 LLM 结合，主要作用是减少幻觉、提升答案准确性。",
    },
    {
        "user_input": "向量数据库有哪些常见产品？",
        "retrieved_contexts": [
            "向量数据库用于存储和检索高维向量嵌入。常见产品包括 Pinecone、Weaviate、Milvus 和 Chroma。"
            "它们支持近似最近邻（ANN）搜索，适合语义相似度检索。",
        ],
        "response": "常见的向量数据库产品有 Pinecone、Weaviate、Milvus 和 Chroma，均支持近似最近邻搜索。",
        "reference": "常见向量数据库包括 Pinecone、Weaviate、Milvus、Chroma，支持 ANN 搜索。",
    },
    {
        "user_input": "Transformer 架构是什么时候提出的？有什么创新？",
        "retrieved_contexts": [
            "Transformer 架构由 Vaswani 等人在 2017 年的论文《Attention Is All You Need》中提出。"
            "它完全基于注意力机制，摒弃了循环和卷积结构，在机器翻译任务上取得了突破性成果。",
            "提示工程是设计和优化输入提示以引导 LLM 产生期望输出的技术。常见技巧包括少样本提示和思维链提示。",
        ],
        "response": "Transformer 架构于 2017 年提出。其核心创新是完全基于注意力机制，去掉了传统的 RNN 和 CNN 结构。",
        "reference": "Transformer 于 2017 年提出，核心创新是纯注意力机制，放弃了 RNN 和 CNN 结构。",
    },
    {
        "user_input": "嵌入模型的作用是什么？有哪些常用模型？",
        "retrieved_contexts": [
            "嵌入模型将文本映射到稠密向量空间，使语义相近的文本在向量空间中距离较近。"
            "常用的嵌入模型包括 OpenAI text-embedding-ada-002、BGE 系列和 E5 系列模型。",
        ],
        "response": "嵌入模型的作用是将文本转换为向量，使语义相似的文本在向量空间中更接近。常用模型有 ada-002、BGE、E5 系列。",
        "reference": "嵌入模型将文本映射为向量，用于语义检索，代表性模型有 ada-002、BGE、E5 系列。",
    },
]

print(f"评估样本: {len(EVAL_SAMPLES)} 条")

In [ ]:
# ========== 框架选型 ==========
frameworks = pd.DataFrame([
    ["RAGAS",    "RAG 专用评估",          "✅ 原生支持", "✅ 支持",  "❌ (DataFrame)", "简单", "⭐⭐⭐⭐⭐", "快速原型、LangChain 生态"],
    ["TruLens",  "LLM App 全链路追踪",    "部分支持",    "❌ 需 GT",  "✅ 内置 UI",     "中等", "⭐⭐⭐",   "企业级 trace 监控"],
    ["DeepEval", "通用 LLM 测试框架",     "✅ 支持",     "✅ 支持",  "✅ 内置 UI",     "简单", "⭐⭐⭐⭐",  "RAG + Agent 统一评估"],
    ["Langfuse", "开源 LLM 可观测性平台", "❌ 不负责评估","❌ 不负责", "✅ 内置 UI",     "简单", "⭐⭐⭐⭐",  "Trace/Prompt管理/Metrics"],
], columns=["框架", "定位", "四大RAG指标", "无监督评估", "UI Dashboard", "CI集成", "社区", "推荐场景"])

print("评估框架 + 可观测性工具选型")
print("=" * 120)
print(frameworks.to_string(index=False))
print("=" * 120)
print("\n推荐组合：RAGAS 算分 + Langfuse 全链路 trace + 自建诊断决策树")

---
## 关键 Takeaway

1. **四个指标缺一不可** — 只看 Faithfulness 会漏掉检索问题，只看 Recall 会漏掉幻觉问题
2. **Binary ≠ Weighted** — RAGAS 的 Weighted Precision 与手动 Binary 版有 10-15% 偏差
3. **Jaccard 代理不可用** — Answer Relevancy 必须用 LLM 反向生成 + Embedding 语义相似度
4. **优化顺序有讲究** — Recall → Faithfulness → Precision → Relevancy（违反顺序 = 沙子上盖房）
5. **不只看均值** — 均值 0.80 可能藏着 30% 的样本得 0 分
6. **NLI 三分类** — Entailment/Contradiction/Neutral；PARTIALLY_SUPPORTED 三分法比二分类更精细
7. **TestsetGenerator** — 从文档自动生成评估集；官方全自动 vs 自定义逐Chunk两种方案
8. **Langfuse + RAGAS** — 离线算分 + 在线可观测 = 评估全覆盖

> 详细解析见 `RAG_Evaluation_Deep_Dive.md`

In [ ]:
# ========== 阈值配置 ==========
THRESHOLDS = {
    "context_recall":    0.75,
    "context_precision": 0.65,
    "faithfulness":      0.80,
    "answer_relevancy":  0.80,
}

# ========== 诊断引擎 ==========
REMEDIATIONS = {
    "context_recall": {
        "label": "Context Recall（上下文召回率）",
        "causes": ["top_k 太小", "chunk_size 太大", "embedding 模型召回差"],
        "fixes": [
            "增大 top_k（3 → 6 或更大）",
            "缩小 chunk_size（1000 → 512），增加 overlap",
            "加入 BM25 混合检索",
            "换更强的 Embedding 模型（如 bge-large-zh-v1.5）",
        ],
    },
    "context_precision": {
        "label": "Context Precision（上下文精确率）",
        "causes": ["检索噪声多", "top_k 太大", "缺少 Reranker"],
        "fixes": [
            "加 Reranker（如 Cohere Rerank / BGE-Reranker）",
            "提高相似度阈值（score_threshold: 0.75+）",
            "减小 top_k（减少噪声引入）",
            "清洗知识库，去除低质量文档",
        ],
    },
    "faithfulness": {
        "label": "Faithfulness（忠实度）",
        "causes": ["System Prompt 约束太弱", "temperature 太高", "Recall 低导致 LLM 补全"],
        "fixes": [
            "强化 System Prompt（'只使用提供的上下文回答'）",
            "降低 temperature 到 0",
            "先检查并修复 Context Recall（没材料 → 必然编造）",
            "要求 LLM 在每句话后标注引用来源",
        ],
    },
    "answer_relevancy": {
        "label": "Answer Relevancy（答案相关性）",
        "causes": ["问题模糊", "Prompt 没引导直接回答", "检索偏向无关内容"],
        "fixes": [
            "Query 重写（让问题更精确）",
            "HyDE：先生成假设答案再用答案 embedding 检索",
            "优化 Response Prompt（'首句直接回答，不要铺垫'）",
            "多查询检索（多个变体 query 并行，结果合并去重）",
        ],
    },
}


def generate_diagnostics(metrics: dict) -> list:
    """根据指标与阈值对比，生成优先级排序的诊断建议"""
    diag = []
    for key, info in REMEDIATIONS.items():
        score = metrics.get(key, 0.0)
        threshold = THRESHOLDS[key]
        gap = threshold - score
        passed = score >= threshold
        diag.append({
            "metric": info["label"],
            "key": key,
            "score": score,
            "threshold": threshold,
            "gap": gap,
            "passed": passed,
            "causes": info["causes"],
            "fixes": info["fixes"],
        })
    # 按缺口从大到小排序（最严重的排最前）
    diag.sort(key=lambda x: x["gap"], reverse=True)
    return diag


# ========== 四象限判断 ==========
def classify_quadrant(precision: float, recall: float) -> str:
    """根据 Precision 和 Recall 判断四象限"""
    p_ok = precision >= 0.65
    r_ok = recall >= 0.75
    if p_ok and r_ok:
        return "右上：检索器健康 → 重点查 Generator（Faithfulness + Relevancy）"
    elif p_ok and not r_ok:
        return "左上：精密但不全 → 增大 top_k、调小 chunk_size、换 embedding"
    elif not p_ok and r_ok:
        return "右下：完整但噪声多 → 加 Reranker、提高相似度阈值、减小 top_k"
    else:
        return "左下(🔴最危险)：双低 → 优先修 embedding + chunking 基础策略"


# ========== 示例诊断 ==========
sample_metrics = {
    "context_recall": 0.50,
    "context_precision": 0.50,
    "faithfulness": 0.67,
    "answer_relevancy": 0.85,
}

diag = generate_diagnostics(sample_metrics)
quadrant = classify_quadrant(
    sample_metrics["context_precision"],
    sample_metrics["context_recall"]
)

print("=" * 65)
print("RAG 诊断报告")
print("=" * 65)
print(f"\n📍 四象限定位: {quadrant}\n")

print("📊 指标详情:")
print(f"  {'指标':<35} {'分数':>6} {'阈值':>6} {'状态':>6}")
print("-" * 60)
for d in diag:
    status = "✅" if d["passed"] else "⚠️"
    print(f"  {d['metric']:<35} {d['score']:>6.2f} {d['threshold']:>6.2f}  {status:>4}")

print(f"\n🔧 修复建议（按优先级排序）:")
for i, d in enumerate(diag, 1):
    if d["passed"]:
        continue
    print(f"\n  [{i}] {d['metric']} (缺口: {d['gap']:+.2f})")
    print(f"      可能原因: {', '.join(d['causes'])}")
    print(f"      修复方向:")
    for fix in d["fixes"]:
        print(f"        → {fix}")

print("\n" + "=" * 65)
print("📋 优化优先级: Recall → Faithfulness → Precision → Relevancy")
print("=" * 65)

---
# Part 5 · 速查表

In [ ]:
import pandas as pd

# ========== 指标总览 ==========
overview = pd.DataFrame([
    ["Context Recall",    "该找的都找到了吗",  "检索-完整", "\|GT∩Ret\| / \|GT\| (Binary)",               "≥ 0.75", "增大 top_k、调 chunk、换 embedding"],
    ["Context Precision", "找的都相关吗",      "检索-精准", "∑P@i·rel / ∑rel (Weighted, RAGAS)",            "≥ 0.65", "加 Reranker、提相似度阈值"],
    ["Faithfulness",      "有没有瞎编",        "生成-可信", "被支撑 Claim / 总 Claim (LLM-as-Judge)",         "≥ 0.80", "严格 System Prompt、降 temperature"],
    ["Answer Relevancy",  "答的切题吗",        "生成-精准", "mean(cos(Q, Q'_i)) (反向问题+Embedding)",       "≥ 0.80", "Query 重写、HyDE、优化 prompt"],
    ["HitRate@k",         "至少找到一个了吗",   "检索-排序", "1\[top-k 命中\]",                                "≥ 0.90", "增大 k 或改善检索"],
    ["MRR",               "第一个排第几",      "检索-排序", "mean(1/rank)",                                   "≥ 0.60", "改善排序模型"],
    ["NDCG",              "高相关排前面了吗",   "检索-排序", "DCG / IDCG",                                     "≥ 0.60", "多级标注 + 排序优化"],
], columns=["指标", "核心问题", "维度", "公式简述", "参考阈值", "低分修复方向"])

print("RAG 评估指标速查表")
print("=" * 120)
print(overview.to_string(index=False))
print("=" * 120)

In [ ]:
# ========== 优化策略速查 ==========
fixes = pd.DataFrame([
    ["Context Recall 低",       "增大 top_k",                   "retriever = vectorstore.as_retriever(search_kwargs={'k': 6})", "↑↑", "检索时间增加"],
    ["Context Recall 低",       "缩小 chunk_size",              "RecursiveCharacterTextSplitter(chunk_size=512)",               "↑↑", "检索数量增大"],
    ["Context Recall 低",       "混合检索",                     "EnsembleRetriever([bm25, vector])",                           "↑↑↑", "复杂度增加"],
    ["Context Recall 低",       "换 embedding",                 "text-embedding-3-large / bge-large-zh-v1.5",                  "↑↑↑", "API 费用"],
    ["Context Precision 低",    "加 Reranker",                  "CohereRerank(model='rerank-multilingual-v3.0')",               "↑↑↑", "额外 API 调用"],
    ["Context Precision 低",    "提高相似度阈值",               "score_threshold=0.75",                                        "↑↑", "可能降低 Recall"],
    ["Faithfulness 低",         "强化 System Prompt",           "'只使用参考文档中的信息'",                                   "↑↑", "—"],
    ["Faithfulness 低",         "降低 temperature",             "temperature=0.0",                                             "↑", "降低创造性"],
    ["Faithfulness 低",         "先修 Recall",                  "Recall 低 → LLM 没材料只好编",                                "根本修复", "—"],
    ["Answer Relevancy 低",     "Query 重写",                   "'将问题改写为更清晰的检索查询'",                              "↑", "—"],
    ["Answer Relevancy 低",     "HyDE",                         "先让 LLM 生成假设答案再检索",                                 "↑↑", "额外 LLM 调用"],
    ["Answer Relevancy 低",     "优化 Response Prompt",         "'首句必须直接给出答案'",                                     "↑", "—"],
], columns=["问题", "策略", "代码/操作", "效果", "代价"])

print("优化策略速查表")
print("=" * 120)
print(fixes.to_string(index=False))
print("=" * 120)

In [ ]:
# ========== 框架选型 ==========
frameworks = pd.DataFrame([
    ["RAGAS",    "RAG 专用评估",          "✅ 原生支持", "✅ 支持",  "❌ (DataFrame)", "简单", "⭐⭐⭐⭐⭐", "快速原型、LangChain 生态"],
    ["TruLens",  "LLM App 全链路追踪",    "部分支持",    "❌ 需 GT",  "✅ 内置 UI",     "中等", "⭐⭐⭐",   "企业级 trace 监控"],
    ["DeepEval", "通用 LLM 测试框架",     "✅ 支持",     "✅ 支持",  "✅ 内置 UI",     "简单", "⭐⭐⭐⭐",  "RAG + Agent 统一评估"],
], columns=["框架", "定位", "四大RAG指标", "无监督评估", "UI Dashboard", "CI集成", "社区", "推荐场景"])

print("评估框架选型")
print("=" * 120)
print(frameworks.to_string(index=False))
print("=" * 120)
print("\n推荐组合：RAGAS 算分 + LangSmith/TruLens trace 看板 + 自建诊断决策树")

---
## 关键 Takeaway

1. **四个指标缺一不可** — 只看 Faithfulness 会漏掉检索问题，只看 Recall 会漏掉幻觉问题
2. **Binary ≠ Weighted** — RAGAS 的 Weighted Precision 与手动 Binary 版有 10-15% 偏差
3. **Jaccard 代理不可用** — Answer Relevancy 必须用 LLM 反向生成 + Embedding 语义相似度
4. **优化顺序有讲究** — Recall → Faithfulness → Precision → Relevancy（违反顺序 = 沙子上盖房）
5. **不只看均值** — 均值 0.80 可能藏着 30% 的样本得 0 分

> 详细解析见 `RAG_Evaluation_Deep_Dive.md`